# Load packages and libraries

In [3]:
.libPaths()
assign(".lib.loc", "/home/manuel.tardaguila/conda_envs/multiome_NEW_downstream_analysis/lib/R/library", envir = environment(.libPaths))
.libPaths()
# sessionInfo()


suppressMessages(library(Seurat))
suppressMessages(library(Signac))
suppressMessages(library(dplyr)) 
suppressMessages(library(ggplot2)) 
suppressMessages(library(Matrix)) 
suppressMessages(library(data.table)) 
suppressMessages(library(ggpubr)) 
suppressMessages(library(ggplot2))
suppressMessages(library(pheatmap))
suppressMessages(library(presto))
suppressMessages(library("qlcMatrix"))
suppressMessages(library("cowplot"))
suppressMessages(library("RColorBrewer"))
suppressMessages(library("plyr"))
suppressMessages(library("forcats"))
suppressMessages(library('ggeasy'))
suppressMessages(library('dplyr'))
suppressMessages(library("svglite"))
suppressMessages(library("ape"))
suppressMessages(library("ggforce"))
suppressMessages(library("tidyr"))
suppressMessages(library("edgeR"))
suppressMessages(library("apeglm"))
suppressMessages(library("DESeq2"))
suppressMessages(library("tibble")) 
suppressMessages(library(future))
suppressMessages(library(future))


library("optparse")
suppressMessages(library("splitstackshape")) 
suppressMessages(library("ggupset"))

library("org.Mm.eg.db", lib.loc="/home/manuel.tardaguila/R/x86_64-pc-linux-gnu-library/4.1/")

suppressMessages(library("ggrepel"))



[1] "/home/manuel.tardaguila/conda_envs/multiome_NEW_downstream_analysis/lib/R/library"

[1] "/home/manuel.tardaguila/conda_envs/multiome_NEW_downstream_analysis/lib/R/library"

# Read in featureCounts files and transform to matrix structure

In [4]:
setwd("/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025/")

list_files<-dir()

list_files_sel<-list_files[grep("_Quant.isoforms.results$", list_files)]

str(list_files_sel)
cat("\n")

sample_vector<-gsub("_Quant.isoforms.results$","",list_files_sel)

str(sample_vector)
cat("\n")

df<-as.data.frame(cbind(list_files_sel,sample_vector))
colnames(df)<-c('file','sample')

df$file<-paste("/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025/",df$file, sep='')


str(df)
cat("\n")




 chr [1:30] "LM411_R1_t0_Quant.isoforms.results" ...

 chr [1:30] "LM411_R1_t0" "LM411_R1_t1" "LM411_R1_t2" "LM411_R1_t3" ...

'data.frame':	30 obs. of  2 variables:
 $ file  : chr  "/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025/LM411_R1_t0_Quant.isoforms.results" "/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025/LM411_R1_t1_Quant.isoforms.results" "/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025/LM411_R1_t2_Quant.isoforms.results" "/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025/LM411_R1_t3_Quant.isoforms.results" ...
 $ sample: chr  "LM411_R1_t0" "LM411_R1_t1" "LM411_R1_t2" "LM411_R1_t3" ...



In [5]:
DEBUG<-0

# Initialize a variable to store the merged count matrix (optional, but good practice)
all_raw_counts <- NULL 


for(i in 1:dim(df)[1]){

    sample_array_sel<-df$sample[i]

    cat("-------------------------->\t")
    cat(sprintf(as.character(sample_array_sel)))
    cat("\n")

    raw_counts_file<-df$file[i]

    raw_counts_df<-read.table(file=raw_counts_file,sep="\t", header=TRUE, stringsAsFactors = FALSE)
    #colnames(raw_counts_df)[8]<-sample_array_sel


    if(DEBUG == 1){
        
        cat("raw_counts_df_0\n")
        str(raw_counts_df)
        cat("\n")
    }


    indx_sel<-which(colnames(raw_counts_df)%in%c('gene_id','transcript_id','expected_count'))


    raw_counts_df_subset<-raw_counts_df[,indx_sel]

    

    if(DEBUG == 1){
        
        cat("raw_counts_df_subset_0\n")
        str(raw_counts_df_subset)
        cat("\n")
    }

    colnames(raw_counts_df_subset)[which(colnames(raw_counts_df_subset) == 'expected_count')]<-sample_array_sel

    # 3. Initialize or Merge the count matrix
    if (is.null(all_raw_counts)) {
        # If this is the first sample, initialize the matrix
        all_raw_counts <- raw_counts_df_subset
        } else {
            # For subsequent samples, merge the new count column 
            # using 'transcript_id' as the key to align the rows.
            all_raw_counts <- merge(
                x = all_raw_counts, 
                y = raw_counts_df_subset, 
                by.x = c('gene_id','transcript_id'), 
                by.y = c('gene_id','transcript_id'),
                all = TRUE # Keeps all Gene IDs, even if a sample has a count of 0 for a gene.
            )
        }#if (is.null(all_raw_counts)

    if(DEBUG == 1){
        
        cat("all_raw_counts_0\n")
        str(all_raw_counts)
        cat("\n")
    }

    
}#i in 1:length(sample_array)

-------------------------->	LM411_R1_t0
-------------------------->	LM411_R1_t1
-------------------------->	LM411_R1_t2
-------------------------->	LM411_R1_t3
-------------------------->	LM411_R1_t4
-------------------------->	LM411_R2_t0
-------------------------->	LM411_R2_t1
-------------------------->	LM411_R2_t2
-------------------------->	LM411_R2_t3
-------------------------->	LM411_R2_t4
-------------------------->	LM411_R3_t0
-------------------------->	LM411_R3_t1
-------------------------->	LM411_R3_t2
-------------------------->	LM411_R3_t3
-------------------------->	LM411_R3_t4
-------------------------->	LM511_R1_t0
-------------------------->	LM511_R1_t1
-------------------------->	LM511_R1_t2
-------------------------->	LM511_R1_t3
-------------------------->	LM511_R1_t4
-------------------------->	LM511_R2_t0
-------------------------->	LM511_R2_t1
-------------------------->	LM511_R2_t2
-------------------------->	LM511_R2_t3
-------------------------->	LM511_R2_t4


In [6]:
cat("all_raw_counts_0\n")
str(all_raw_counts)
cat("\n")

all_raw_counts_0
'data.frame':	278396 obs. of  32 variables:
 $ gene_id      : chr  "ENSMUSG00000000001" "ENSMUSG00000000003" "ENSMUSG00000000003" "ENSMUSG00000000028" ...
 $ transcript_id: chr  "ENSMUST00000000001" "ENSMUST00000000003" "ENSMUST00000114041" "ENSMUST00000000028" ...
 $ LM411_R1_t0  : num  5383.8 0 0 55.6 0 ...
 $ LM411_R1_t1  : num  8039 0 0 763 144 ...
 $ LM411_R1_t2  : num  6724 0 0 1808 137 ...
 $ LM411_R1_t3  : num  6246 0 0 1742 265 ...
 $ LM411_R1_t4  : num  10693 0 0 2975 316 ...
 $ LM411_R2_t0  : num  6593.9 0 0 50.9 0 ...
 $ LM411_R2_t1  : num  7304 0 0 568 121 ...
 $ LM411_R2_t2  : num  7481 0 0 1920 152 ...
 $ LM411_R2_t3  : num  6583 0 0 1683 116 ...
 $ LM411_R2_t4  : num  8048 0 0 2040 419 ...
 $ LM411_R3_t0  : num  6289.8 0 0 58.6 0 ...
 $ LM411_R3_t1  : num  8712.5 0 0 687.2 40.9 ...
 $ LM411_R3_t2  : num  6660 0 0 1626 147 ...
 $ LM411_R3_t3  : num  7215 0 0 1842 181 ...
 $ LM411_R3_t4  : num  8932 0 0 2473 243 ...
 $ LM511_R1_t0  : num  4556 0 0 40 0 ..

In [7]:
all_raw_counts$transcript_id<-gsub("\\..+$","",all_raw_counts$transcript_id)
all_raw_counts$gene_id<-gsub("\\..+$","",all_raw_counts$gene_id)

str(all_raw_counts)
cat("\n")

'data.frame':	278396 obs. of  32 variables:
 $ gene_id      : chr  "ENSMUSG00000000001" "ENSMUSG00000000003" "ENSMUSG00000000003" "ENSMUSG00000000028" ...
 $ transcript_id: chr  "ENSMUST00000000001" "ENSMUST00000000003" "ENSMUST00000114041" "ENSMUST00000000028" ...
 $ LM411_R1_t0  : num  5383.8 0 0 55.6 0 ...
 $ LM411_R1_t1  : num  8039 0 0 763 144 ...
 $ LM411_R1_t2  : num  6724 0 0 1808 137 ...
 $ LM411_R1_t3  : num  6246 0 0 1742 265 ...
 $ LM411_R1_t4  : num  10693 0 0 2975 316 ...
 $ LM411_R2_t0  : num  6593.9 0 0 50.9 0 ...
 $ LM411_R2_t1  : num  7304 0 0 568 121 ...
 $ LM411_R2_t2  : num  7481 0 0 1920 152 ...
 $ LM411_R2_t3  : num  6583 0 0 1683 116 ...
 $ LM411_R2_t4  : num  8048 0 0 2040 419 ...
 $ LM411_R3_t0  : num  6289.8 0 0 58.6 0 ...
 $ LM411_R3_t1  : num  8712.5 0 0 687.2 40.9 ...
 $ LM411_R3_t2  : num  6660 0 0 1626 147 ...
 $ LM411_R3_t3  : num  7215 0 0 1842 181 ...
 $ LM411_R3_t4  : num  8932 0 0 2473 243 ...
 $ LM511_R1_t0  : num  4556 0 0 40 0 ...
 $ LM511_R1_t1 

In [8]:
multiVals <- function(x) paste(x,collapse=";")

In [9]:
all_raw_counts$gene_name <- mapIds(org.Mm.eg.db, keys=all_raw_counts$gene_id, keytype="ENSEMBL",column="SYMBOL", multiVals=multiVals)

str(all_raw_counts)
cat("\n")

'select()' returned 1:many mapping between keys and columns



'data.frame':	278396 obs. of  33 variables:
 $ gene_id      : chr  "ENSMUSG00000000001" "ENSMUSG00000000003" "ENSMUSG00000000003" "ENSMUSG00000000028" ...
 $ transcript_id: chr  "ENSMUST00000000001" "ENSMUST00000000003" "ENSMUST00000114041" "ENSMUST00000000028" ...
 $ LM411_R1_t0  : num  5383.8 0 0 55.6 0 ...
 $ LM411_R1_t1  : num  8039 0 0 763 144 ...
 $ LM411_R1_t2  : num  6724 0 0 1808 137 ...
 $ LM411_R1_t3  : num  6246 0 0 1742 265 ...
 $ LM411_R1_t4  : num  10693 0 0 2975 316 ...
 $ LM411_R2_t0  : num  6593.9 0 0 50.9 0 ...
 $ LM411_R2_t1  : num  7304 0 0 568 121 ...
 $ LM411_R2_t2  : num  7481 0 0 1920 152 ...
 $ LM411_R2_t3  : num  6583 0 0 1683 116 ...
 $ LM411_R2_t4  : num  8048 0 0 2040 419 ...
 $ LM411_R3_t0  : num  6289.8 0 0 58.6 0 ...
 $ LM411_R3_t1  : num  8712.5 0 0 687.2 40.9 ...
 $ LM411_R3_t2  : num  6660 0 0 1626 147 ...
 $ LM411_R3_t3  : num  7215 0 0 1842 181 ...
 $ LM411_R3_t4  : num  8932 0 0 2473 243 ...
 $ LM511_R1_t0  : num  4556 0 0 40 0 ...
 $ LM511_R1_t1 

## Remove all NA gene symbol

In [10]:
indx.NA<-which(all_raw_counts$gene_name == "NA")

str(indx.NA)
cat("\n")

all_raw_counts<-all_raw_counts[-indx.NA,]

str(all_raw_counts)
cat("\n")

 int [1:129696] 933 934 935 936 6693 8622 8623 12623 14840 14841 ...

'data.frame':	148700 obs. of  33 variables:
 $ gene_id      : chr  "ENSMUSG00000000001" "ENSMUSG00000000003" "ENSMUSG00000000003" "ENSMUSG00000000028" ...
 $ transcript_id: chr  "ENSMUST00000000001" "ENSMUST00000000003" "ENSMUST00000114041" "ENSMUST00000000028" ...
 $ LM411_R1_t0  : num  5383.8 0 0 55.6 0 ...
 $ LM411_R1_t1  : num  8039 0 0 763 144 ...
 $ LM411_R1_t2  : num  6724 0 0 1808 137 ...
 $ LM411_R1_t3  : num  6246 0 0 1742 265 ...
 $ LM411_R1_t4  : num  10693 0 0 2975 316 ...
 $ LM411_R2_t0  : num  6593.9 0 0 50.9 0 ...
 $ LM411_R2_t1  : num  7304 0 0 568 121 ...
 $ LM411_R2_t2  : num  7481 0 0 1920 152 ...
 $ LM411_R2_t3  : num  6583 0 0 1683 116 ...
 $ LM411_R2_t4  : num  8048 0 0 2040 419 ...
 $ LM411_R3_t0  : num  6289.8 0 0 58.6 0 ...
 $ LM411_R3_t1  : num  8712.5 0 0 687.2 40.9 ...
 $ LM411_R3_t2  : num  6660 0 0 1626 147 ...
 $ LM411_R3_t3  : num  7215 0 0 1842 181 ...
 $ LM411_R3_t4  : num  8932 0 0

## Force all NAs 0

In [11]:
all_raw_counts[is.na(all_raw_counts)] <- 0

## Convert into a DESeq2 count matrix

In [12]:
all_raw_counts_matrix<-as.matrix(all_raw_counts[,-c(which(colnames(all_raw_counts) == 'transcript_id'),
                                                    which(colnames(all_raw_counts) == 'gene_id'),
                                                    which(colnames(all_raw_counts) == 'gene_name'))])

row.names(all_raw_counts_matrix)<-all_raw_counts$transcript_id


cat("all_raw_counts_matrix_0\n")
str(all_raw_counts_matrix)
cat("\n")

all_raw_counts_matrix_0
 num [1:148700, 1:30] 5383.8 0 0 55.6 0 ...
 - attr(*, "dimnames")=List of 2
  ..$ : chr [1:148700] "ENSMUST00000000001" "ENSMUST00000000003" "ENSMUST00000114041" "ENSMUST00000000028" ...
  ..$ : chr [1:30] "LM411_R1_t0" "LM411_R1_t1" "LM411_R1_t2" "LM411_R1_t3" ...



# Keep to integers

In [13]:
all_raw_counts_matrix <- round(all_raw_counts_matrix, digits = 0)


cat("all_raw_counts_matrix_1\n")
str(all_raw_counts_matrix)
cat("\n")

all_raw_counts_matrix_1
 num [1:148700, 1:30] 5384 0 0 56 0 ...
 - attr(*, "dimnames")=List of 2
  ..$ : chr [1:148700] "ENSMUST00000000001" "ENSMUST00000000003" "ENSMUST00000114041" "ENSMUST00000000028" ...
  ..$ : chr [1:30] "LM411_R1_t0" "LM411_R1_t1" "LM411_R1_t2" "LM411_R1_t3" ...



## Check some expression values

In [15]:
all_raw_counts_matrix[which(row.names(all_raw_counts_matrix)%in%c('ENSMUST00000186857','ENSMUST00000186574')),]

,LM411_R1_t0,LM411_R1_t1,LM411_R1_t2,LM411_R1_t3,LM411_R1_t4,LM411_R2_t0,LM411_R2_t1,LM411_R2_t2,LM411_R2_t3,LM411_R2_t4,⋯,LM511_R2_t0,LM511_R2_t1,LM511_R2_t2,LM511_R2_t3,LM511_R2_t4,LM511_R3_t0,LM511_R3_t1,LM511_R3_t2,LM511_R3_t3,LM511_R3_t4
ENSMUST00000186574,9693,7733,8650,7917,11219,12239,6470,9681,8265,9042,⋯,10373,6146,10496,7979,11790,7528,5996,9398,8613,8447
ENSMUST00000186857,11002,6971,9415,9844,13180,12900,6301,10912,9603,10523,⋯,11222,6198,12192,11159,15192,9388,6531,10362,12757,11121


## Count matrix

In [16]:
count_matrix<-all_raw_counts_matrix

cat("count_matrix_0\n")
cat(str(count_matrix))
cat("\n")

count_matrix_0
 num [1:148700, 1:30] 5384 0 0 56 0 ...
 - attr(*, "dimnames")=List of 2
  ..$ : chr [1:148700] "ENSMUST00000000001" "ENSMUST00000000003" "ENSMUST00000114041" "ENSMUST00000000028" ...
  ..$ : chr [1:30] "LM411_R1_t0" "LM411_R1_t1" "LM411_R1_t2" "LM411_R1_t3" ...



# Create the colData object

In [17]:
samples<-colnames(count_matrix)

cat("samples/n")
cat(str(samples))


samples/n chr [1:30] "LM411_R1_t0" "LM411_R1_t1" "LM411_R1_t2" "LM411_R1_t3" ...


In [18]:
treat<-gsub("_.+$","",samples)

cat("treat/n")
cat(str(treat))


treat/n chr [1:30] "LM411" "LM411" "LM411" "LM411" "LM411" "LM411" "LM411" "LM411" ...


In [19]:
batch<-gsub("^[^_]+_","",samples)
batch<-gsub("_.+$","",batch)


cat("batch/n")
cat(str(batch))


batch/n chr [1:30] "R1" "R1" "R1" "R1" "R1" "R2" "R2" "R2" "R2" "R2" "R3" "R3" ...


In [20]:
time<-gsub("^[^_]+_[^_]+_","",samples)

cat("time/n")
cat(str(time))


time/n chr [1:30] "t0" "t1" "t2" "t3" "t4" "t0" "t1" "t2" "t3" "t4" "t0" "t1" ...


In [21]:
colData<-as.data.frame(cbind(samples,treat,batch,time))

colnames(colData)<-c('sample','treat','batch','time')

cat("colData_0/n")
cat(str(colData))


colData_0/n'data.frame':	30 obs. of  4 variables:
 $ sample: chr  "LM411_R1_t0" "LM411_R1_t1" "LM411_R1_t2" "LM411_R1_t3" ...
 $ treat : chr  "LM411" "LM411" "LM411" "LM411" ...
 $ batch : chr  "R1" "R1" "R1" "R1" ...
 $ time  : chr  "t0" "t1" "t2" "t3" ...


## factorize treat

In [22]:
colData$treat<-factor(colData$treat)
colData$treat<-relevel(colData$treat, ref='LM411') ### Nothing works with ordered factors

colData$time<-factor(colData$time)
colData$time<-relevel(colData$time, ref='t0') ### Nothing works with ordered factors

cat("colData_1/n")
cat(str(colData))

colData_1/n'data.frame':	30 obs. of  4 variables:
 $ sample: chr  "LM411_R1_t0" "LM411_R1_t1" "LM411_R1_t2" "LM411_R1_t3" ...
 $ treat : Factor w/ 2 levels "LM411","LM511": 1 1 1 1 1 1 1 1 1 1 ...
 $ batch : chr  "R1" "R1" "R1" "R1" ...
 $ time  : Factor w/ 5 levels "t0","t1","t2",..: 1 2 3 4 5 1 2 3 4 5 ...


# DESeq2

## Create the DESeqDataSetFromMatrix object

In [23]:
dds <- DESeqDataSetFromMatrix(countData = count_matrix,
                              colData = colData,
                              design = ~ treat + time)

converting counts to integer mode



In [24]:
cat("dds_0/n")
#cat(str(dds))

dds_0/n

## Filter out transcripts with fewer than  total raw counts

In [25]:
keep <- rowSums(counts(dds)) >= 10

filtered_out<-keep[which(keep == "FALSE")]

cat(str(filtered_out))

 Named logi [1:76745] FALSE FALSE FALSE FALSE FALSE FALSE ...
 - attr(*, "names")= chr [1:76745] "ENSMUST00000000003" "ENSMUST00000114041" "ENSMUST00000132294" "ENSMUST00000140716" ...


In [27]:
which(names(filtered_out)%in%c('ENSMUST00000186857','ENSMUST00000186574'))

integer(0)

In [28]:
dds <- dds[keep,]

## estimateSizeFactors----------------------------------


In [29]:
dds<-estimateSizeFactors(dds)

## LRT test from the reduced model

In [30]:
dds_lrt <- DESeq(dds, 
                 test = "LRT", 
                 reduced = ~ time)

using pre-existing size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



## Extract results

In [31]:
possible_contrasts<-colnames(dds_lrt@modelMatrix)[-1] # -1 because 1 is the Intercept term

possible_contrasts

[1] "treat_LM511_vs_LM411" "time_t1_vs_t0"        "time_t2_vs_t0"       
[4] "time_t3_vs_t0"        "time_t4_vs_t0"

In [32]:
resultsNames(dds_lrt)

[1] "Intercept"            "treat_LM511_vs_LM411" "time_t1_vs_t0"       
[4] "time_t2_vs_t0"        "time_t3_vs_t0"        "time_t4_vs_t0"

In [33]:
tmp_results<-results(dds_lrt, test= "Wald", name=possible_contrasts[1], independentFiltering=FALSE)
        
#### expand LogFC #########################################

tmp_results <- lfcShrink(dds_lrt, 
                         coef = possible_contrasts[1],
                         res=tmp_results,
                         type = "apeglm")


cat("tmp_results_0/n")
#cat(str(tmp_results))

using 'apeglm' for LFC shrinkage. If used in published research, please cite:
    Zhu, A., Ibrahim, J.G., Love, M.I. (2018) Heavy-tailed prior distributions for
    sequence count data: removing the noise and preserving large differences.
    Bioinformatics. https://doi.org/10.1093/bioinformatics/bty895



tmp_results_0/n

In [34]:
 #### obtain data frame #########################################
        
tmp_tb <- as.data.frame(tmp_results %>%
                          data.frame() %>%
                          rownames_to_column(var = "transcript_id") %>%
                          as_tibble() %>%
                          arrange(padj), stringsAsFactors=F)

tmp_tb$contrast<-possible_contrasts[1]

cat("tmp_tb_0")
cat(str(tmp_tb))

tmp_tb_0'data.frame':	71955 obs. of  7 variables:
 $ transcript_id : chr  "ENSMUST00000042603" "ENSMUST00000164993" "ENSMUST00000029275" "ENSMUST00000038160" ...
 $ baseMean      : num  2565 591 11921 1317 382 ...
 $ log2FoldChange: num  2.49 2.39 2.61 2.44 3.08 ...
 $ lfcSE         : num  0.0815 0.1131 0.1251 0.1236 0.1598 ...
 $ pvalue        : num  1.48e-207 3.95e-101 9.86e-99 2.19e-88 5.10e-86 ...
 $ padj          : num  1.06e-202 1.41e-96 2.35e-94 3.90e-84 7.28e-82 ...
 $ contrast      : chr  "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" ...


In [35]:
tmp_tb$minuslog10padj <- -log10(tmp_tb$padj)
      
tmp_tb$abslogfc<-abs(tmp_tb$log2FoldChange)

cat("tmp_tb_0")
cat(str(tmp_tb))

tmp_tb_0'data.frame':	71955 obs. of  9 variables:
 $ transcript_id : chr  "ENSMUST00000042603" "ENSMUST00000164993" "ENSMUST00000029275" "ENSMUST00000038160" ...
 $ baseMean      : num  2565 591 11921 1317 382 ...
 $ log2FoldChange: num  2.49 2.39 2.61 2.44 3.08 ...
 $ lfcSE         : num  0.0815 0.1131 0.1251 0.1236 0.1598 ...
 $ pvalue        : num  1.48e-207 3.95e-101 9.86e-99 2.19e-88 5.10e-86 ...
 $ padj          : num  1.06e-202 1.41e-96 2.35e-94 3.90e-84 7.28e-82 ...
 $ contrast      : chr  "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" ...
 $ minuslog10padj: num  202 95.9 93.6 83.4 81.1 ...
 $ abslogfc      : num  2.49 2.39 2.61 2.44 3.08 ...


In [40]:
DTE_results<-tmp_tb

## Extract normalized counts

In [41]:
nor_counts<-as.matrix(counts(dds, normalized=TRUE))
      
row.names(nor_counts)<-row.names(nor_counts)

cat("nor_counts_0\n")
str(nor_counts)
cat("\n")


nor_counts_0
 num [1:71955, 1:30] 5891.36 61.28 0 2.19 0 ...
 - attr(*, "dimnames")=List of 2
  ..$ : chr [1:71955] "ENSMUST00000000001" "ENSMUST00000000028" "ENSMUST00000096990" "ENSMUST00000115585" ...
  ..$ : chr [1:30] "LM411_R1_t0" "LM411_R1_t1" "LM411_R1_t2" "LM411_R1_t3" ...



# Equivalence ENST to gene name

In [42]:
all_raw_counts_subset<-unique(all_raw_counts[,which(colnames(all_raw_counts)%in%c('gene_name','gene_id','transcript_id'))])

cat("all_raw_counts_subset_0\n")
str(all_raw_counts_subset)
cat("\n")

all_raw_counts_subset_0
'data.frame':	148700 obs. of  3 variables:
 $ gene_id      : chr  "ENSMUSG00000000001" "ENSMUSG00000000003" "ENSMUSG00000000003" "ENSMUSG00000000028" ...
 $ transcript_id: chr  "ENSMUST00000000001" "ENSMUST00000000003" "ENSMUST00000114041" "ENSMUST00000000028" ...
 $ gene_name    : chr  "Gnai3" "Pbsn" "Pbsn" "Cdc45" ...



In [43]:
DTE_results<-merge(by=c("transcript_id"),all_raw_counts_subset,DTE_results, all.y=TRUE)

cat("DTE_results_0\n")
str(DTE_results)
cat("\n")

DTE_results_0
'data.frame':	71955 obs. of  11 variables:
 $ transcript_id : chr  "ENSMUST00000000001" "ENSMUST00000000010" "ENSMUST00000000028" "ENSMUST00000000033" ...
 $ gene_id       : chr  "ENSMUSG00000000001" "ENSMUSG00000020875" "ENSMUSG00000000028" "ENSMUSG00000048583" ...
 $ gene_name     : chr  "Gnai3" "Hoxb9" "Cdc45" "Igf2" ...
 $ baseMean      : num  7334.45 5.142 1268.765 0.387 33.847 ...
 $ log2FoldChange: num  0.16947 0.03455 -0.01901 -0.00109 0.77526 ...
 $ lfcSE         : num  0.0379 0.1313 0.0446 0.1268 0.2084 ...
 $ pvalue        : num  2.93e-06 1.68e-01 6.48e-01 9.24e-01 9.97e-06 ...
 $ padj          : num  0.000152 0.712289 0.999863 0.999863 0.000449 ...
 $ contrast      : chr  "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" ...
 $ minuslog10padj: num  3.82 1.47e-01 5.94e-05 5.94e-05 3.35 ...
 $ abslogfc      : num  0.16947 0.03455 0.01901 0.00109 0.77526 ...



# SAVE results

In [49]:
setwd("/group/soranzo/manuel.tardaguila/Bulk_RNA_seq/Santos/Reanalysis_2025/EXP1/")

In [50]:
write.table(DTE_results, file="DTE_results.tsv", sep="\t", quote=F, row.names=F)

write.table(nor_counts, file="nor_counts_transcripts.tsv", sep="\t", quote=F)


## Explore results

In [51]:
SIG<-DTE_results[which(DTE_results$minuslog10padj >= 1.3),]

cat("SIG_0\n")
str(SIG)
cat("\n")

SIG_0
'data.frame':	4088 obs. of  11 variables:
 $ transcript_id : chr  "ENSMUST00000000001" "ENSMUST00000000058" "ENSMUST00000000080" "ENSMUST00000000137" ...
 $ gene_id       : chr  "ENSMUSG00000000001" "ENSMUSG00000000058" "ENSMUSG00000000078" "ENSMUSG00000020152" ...
 $ gene_name     : chr  "Gnai3" "Cav2" "Klf6" "Actr2" ...
 $ baseMean      : num  7334.5 33.8 9268.7 15061 3474 ...
 $ log2FoldChange: num  0.169 0.775 0.816 0.092 0.177 ...
 $ lfcSE         : num  0.0379 0.2084 0.1942 0.0303 0.0294 ...
 $ pvalue        : num  2.93e-06 9.97e-06 1.11e-06 1.66e-03 5.15e-10 ...
 $ padj          : num  1.52e-04 4.49e-04 6.39e-05 3.26e-02 6.58e-08 ...
 $ contrast      : chr  "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" "treat_LM511_vs_LM411" ...
 $ minuslog10padj: num  3.82 3.35 4.19 1.49 7.18 ...
 $ abslogfc      : num  0.169 0.775 0.816 0.092 0.177 ...

